In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/신도시공모전/Population Dataset")

print("=== Dataset 폴더 전체 파일 목록 ===\n")

for file in sorted(BASE_DIR.iterdir()):
    if file.is_file():
        print(file.name)

=== Dataset 폴더 전체 파일 목록 ===

01._격자_(4개_시·구).geojson
02._격자_(하남교산).geojson
03._성연령별_거주인구(격자).csv
22._토지이용계획도_(4개_신도시).geojson
23._토지이용계획도_(하남교산).geojson


# 4개 신도시에서 blockType 별로 거주인구 분포를 학습

01_data_preprocessing.ipynb 파일을 통해 확인 해본 결과

- 기존 03._성연령별_거주인구(격자).csv 의 인구 수의 총합은 실제 신도시 거주인구 수를 반영하지 못한다고 판단
- 따라서 4개의 신도시의 **격자 + 토지이용계획도(blockType)** 를 결합해서 **blockType별 거주인구 분포 특성**을 학습하고, 이를 바탕으로 이후 신도시 인구를 더 현실적으로 재분배 하는 코드 파이프라인을 진행

따라서 이 코드에서는 아래와 같은 절차를 따른다.

1. 4개 신도시 격자와 거주인구 데이터를 결합하여 데이터를 결합하여 격자별 총 인구를 계산한다.

2. 4개 신도시 토지이용계획도와 spatial join을 수행하여 각 격자에 'blockType'을 부여한다.

3. blockType별 거주인구 총합과 비중을 계산하여, 토지이용 유형에 따른 인구 분포 특성을 파악한다.

4. 이후 각 신도시의 실제 계획 인구를 기준으로 격자 인구 단위를 재구성한다.

## 4개 신도시의 blockType별 인구 분포를 학습하는 과정

In [4]:
# 필요한 라이브러리 불러오기
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path

In [5]:
BASE_DIR = Path("/content/drive/MyDrive/신도시공모전/Population Dataset")

grid_4city_fp = BASE_DIR / "01._격자_(4개_시·구).geojson"
pop_fp        = BASE_DIR / "03._성연령별_거주인구(격자).csv"
landuse_4_fp  = BASE_DIR / "22._토지이용계획도_(4개_신도시).geojson"

print("파일 존재 여부")
print("grid_4city:", grid_4city_fp.exists())
print("population:", pop_fp.exists())
print("landuse_4 :", landuse_4_fp.exists())

파일 존재 여부
grid_4city: True
population: True
landuse_4 : True


In [7]:
# 4개 도시 격자, 거주인구 데이터, 4개 신도시 토지이용 계획도를 불러온다.
# 격자와 토지이용 데이터는 모두 공간 객체로 다루기 위해 GeoDataFrame으로 불러온다.

grid_4city = gpd.read_file(grid_4city_fp)
pop = pd.read_csv(pop_fp)
landuse_4 = gpd.read_file(landuse_4_fp)

print("grid_4city shape:", grid_4city.shape)
print("pop shape:", pop.shape)
print("landuse_4 shape:", landuse_4.shape)

print("\ngrid_4city columns:")
print(grid_4city.columns.tolist())

print("\npop columns:")
print(pop.columns.tolist())

print("\nlanduse_4 columns:")
print(landuse_4.columns.tolist())

grid_4city shape: (99323, 4)
pop shape: (99323, 21)
landuse_4 shape: (5337, 5)

grid_4city columns:
['std_yr', 'gbn', 'gid', 'geometry']

pop columns:
['gbn', 'gid', 'year', 'm_20g_pop', 'w_20g_pop', 'm_30g_pop', 'w_30g_pop', 'm_40g_pop', 'w_40g_pop', 'm_50g_pop', 'w_50g_pop', 'm_60g_pop', 'w_60g_pop', 'm_70g_pop', 'w_70g_pop', 'm_80g_pop', 'w_80g_pop', 'm_90g_pop', 'w_90g_pop', 'm_100g_pop', 'w_100g_pop']

landuse_4 columns:
['zoneCode', 'zoneName', 'blockName', 'blockType', 'geometry']


In [8]:
# 격자 단위의 총 거주인구 계산

pop_cols = [c for c in pop.columns if c.endswith("_pop")]

pop[pop_cols] = pop[pop_cols].fillna(0)
pop["total_pop"] = pop[pop_cols].sum(axis=1)

print("총 인구 합계:", pop["total_pop"].sum())
display(pop[["gid", "total_pop"]].head())

총 인구 합계: 2183388.0


,gid,total_pop
0,다사581304,0.0
1,다사581305,0.0
2,다사581306,0.0
3,다사582304,0.0
4,다사582305,0.0


In [9]:
## 격자와 거주인구 결합
# 거주인구 데이터의 'gid'를 기준으로 4개 도시 격자와 결합하여,
# 격자 geometyr와 총 거주인구가 함께 포함된 데이터셋을 구성한다.

grid_pop = grid_4city.merge(
    pop[["gid", "total_pop"]],
    on="gid",
    how="left"
)

grid_pop["total_pop"] = grid_pop["total_pop"].fillna(0)

print("결합 후 shape:", grid_pop.shape)
print("인구 결측 개수:", grid_pop["total_pop"].isna().sum())
display(grid_pop[["gid", "gbn", "total_pop"]].head())

결합 후 shape: (99679, 5)
인구 결측 개수: 0


,gid,gbn,total_pop
0,다사581304,경기도 성남시,0.0
1,다사581305,경기도 성남시,0.0
2,다사581306,경기도 성남시,0.0
3,다사582304,경기도 성남시,0.0
4,다사582305,경기도 성남시,0.0


In [10]:
## 좌표계 통일
# Spatial Join을 위해 두 데이터의 좌표계를 동일하게 맞춘다.
# 본 파이프라인에서는 'EPSG:4326' 을 기준으로 처리한다.

if grid_pop.crs is None:
    grid_pop = grid_pop.set_crs("EPSG:4326")
else:
    grid_pop = grid_pop.to_crs("EPSG:4326")

if landuse_4.crs is None:
    landuse_4 = landuse_4.set_crs("EPSG:4326")
else:
    landuse_4 = landuse_4.to_crs("EPSG:4326")

print("grid_pop CRS:", grid_pop.crs)
print("landuse_4 CRS:", landuse_4.crs)

grid_pop CRS: EPSG:4326
landuse_4 CRS: EPSG:4326


In [11]:
## 토지이용계획도와 격자 매핑
# 각 격자에 대응되는 'blockType'을 부여한다.
# 위 코드의 실행 결과로 토지이용 유형별로의 거주인구 분포를 파악하였다.

use_cols = [c for c in ["blockType", "zoneName", "geometry"] if c in landuse_4.columns]

grid_pop_lu = gpd.sjoin(
    grid_pop,
    landuse_4[use_cols],
    how="inner",
    predicate="intersects"
).drop(columns=["index_right"], errors="ignore")

print("매핑 후 shape:", grid_pop_lu.shape)
print("blockType 결측 개수:", grid_pop_lu["blockType"].isna().sum())
print("blockType 종류 수:", grid_pop_lu["blockType"].nunique())

display(grid_pop_lu[["gid", "gbn", "blockType", "total_pop"]].head())

매핑 후 shape: (26603, 7)
blockType 결측 개수: 0
blockType 종류 수: 88


,gid,gbn,blockType,total_pop
1524,다사611327,경기도 성남시,연립,151.0
1524,다사611327,경기도 성남시,도로,151.0
1524,다사611327,경기도 성남시,근린공원,151.0
1525,다사611328,경기도 성남시,연립,7.0
1525,다사611328,경기도 성남시,근린공원,7.0


In [12]:
## 도시별 분리
# 본 코드에서는 4개 도시를 하나로 합치지 않고, 도시별로의 공간적 이질성(개발 구조, 토지이용 구성, 실제 인구 밀도의 차이 등 고려)을 고려하여
# 소비렬로 구분을 진행하였다.

print("도시별 격자 수")
display(grid_pop_lu["gbn"].value_counts())

print("\n도시별 총 인구 합계")
display(grid_pop_lu.groupby("gbn")["total_pop"].sum().sort_values(ascending=False))

도시별 격자 수


,count
gbn,
경기도 화성시,14993
경기도 성남시,6089
경기도 하남시,3826
서울특별시 송파구,1695



도시별 총 인구 합계


,total_pop
gbn,
경기도 화성시,1012887.0
경기도 하남시,515632.0
경기도 성남시,410595.0
서울특별시 송파구,139211.0


In [13]:
## blockType 별 거주인구 총합 및 비중 계산
# 도시별로 'blockType'에 따른 거주인구 총합을 계산
# 어떤 토지 유형이 실제 거주인구를 많이 수용하는지 확인
block_pop_city = (
    grid_pop_lu
    .groupby(["gbn", "blockType"], as_index=False)["total_pop"]
    .sum()
    .rename(columns={"total_pop": "block_pop_sum"})
)

city_total = (
    block_pop_city
    .groupby("gbn", as_index=False)["block_pop_sum"]
    .sum()
    .rename(columns={"block_pop_sum": "city_pop_sum"})
)

block_pop_city = block_pop_city.merge(city_total, on="gbn", how="left")
block_pop_city["block_pop_ratio"] = block_pop_city["block_pop_sum"] / block_pop_city["city_pop_sum"]

display(block_pop_city.head(20))

,gbn,blockType,block_pop_sum,city_pop_sum,block_pop_ratio
0,경기도 성남시,가스공급설비,0.0,410595.0,0.000000
1,경기도 성남시,가압장,26.0,410595.0,0.000063
2,경기도 성남시,경관녹지,1523.0,410595.0,0.003709
3,경기도 성남시,고등학교,2430.0,410595.0,0.005918
4,경기도 성남시,공공공지,44362.0,410595.0,0.108043
5,경기도 성남시,공공청사,0.0,410595.0,0.000000
6,경기도 성남시,공원,8426.0,410595.0,0.020521
7,경기도 성남시,광장,1843.0,410595.0,0.004489
8,경기도 성남시,교통광장,882.0,410595.0,0.002148
9,경기도 성남시,근린공원,16815.0,410595.0,0.040953


In [14]:
for city in sorted(block_pop_city["gbn"].dropna().unique()):
    print(f"\n===== {city} =====")
    display(
        block_pop_city[block_pop_city["gbn"] == city]
        .sort_values("block_pop_sum", ascending=False)
        .head(15)
    )


===== 경기도 성남시 =====


,gbn,blockType,block_pop_sum,city_pop_sum,block_pop_ratio
24,경기도 성남시,아파트,104742.0,410595.0,0.255098
13,경기도 성남시,도로,61748.0,410595.0,0.150387
4,경기도 성남시,공공공지,44362.0,410595.0,0.108043
12,경기도 성남시,단독주택,40619.0,410595.0,0.098927
18,경기도 성남시,보행자전용도로,30405.0,410595.0,0.074051
32,경기도 성남시,완충녹지,18174.0,410595.0,0.044263
9,경기도 성남시,근린공원,16815.0,410595.0,0.040953
6,경기도 성남시,공원,8426.0,410595.0,0.020521
57,경기도 성남시,하천,5813.0,410595.0,0.014158
46,경기도 성남시,중심상업,5759.0,410595.0,0.014026



===== 경기도 하남시 =====


,gbn,blockType,block_pop_sum,city_pop_sum,block_pop_ratio
80,경기도 하남시,아파트,134442.0,515632.0,0.260732
72,경기도 하남시,도로,79252.0,515632.0,0.153699
87,경기도 하남시,완충녹지,52944.0,515632.0,0.102678
74,경기도 하남시,보행자전용도로,40263.0,515632.0,0.078085
102,경기도 하남시,중심상업,33300.0,515632.0,0.064581
71,경기도 하남시,단독주택,31674.0,515632.0,0.061428
84,경기도 하남시,연결녹지,22373.0,515632.0,0.043389
83,경기도 하남시,업무시설,15410.0,515632.0,0.029886
68,경기도 하남시,근린공원,10724.0,515632.0,0.020798
98,경기도 하남시,주상복합,10373.0,515632.0,0.020117



===== 경기도 화성시 =====


,gbn,blockType,block_pop_sum,city_pop_sum,block_pop_ratio
113,경기도 화성시,공동주택,200013.0,1012887.0,0.197468
124,경기도 화성시,도로,182162.0,1012887.0,0.179844
121,경기도 화성시,녹지,93057.0,1012887.0,0.091873
138,경기도 화성시,아파트,91804.0,1012887.0,0.090636
115,경기도 화성시,공원,86509.0,1012887.0,0.085408
111,경기도 화성시,공공공지,72061.0,1012887.0,0.071144
123,경기도 화성시,단독주택,68763.0,1012887.0,0.067888
129,경기도 화성시,보행자전용도로,40160.0,1012887.0,0.039649
165,경기도 화성시,학교,26746.0,1012887.0,0.026406
147,경기도 화성시,일반상업,22791.0,1012887.0,0.022501



===== 서울특별시 송파구 =====


,gbn,blockType,block_pop_sum,city_pop_sum,block_pop_ratio
182,서울특별시 송파구,아파트,37435.0,139211.0,0.268908
175,서울특별시 송파구,도로,24643.0,139211.0,0.177019
168,서울특별시 송파구,공원,12128.0,139211.0,0.087120
185,서울특별시 송파구,완충녹지,11160.0,139211.0,0.080166
194,서울특별시 송파구,준주거용지,9449.0,139211.0,0.067875
178,서울특별시 송파구,보행자전용도로,9247.0,139211.0,0.066424
167,서울특별시 송파구,공공공지,6877.0,139211.0,0.049400
199,서울특별시 송파구,학교,6077.0,139211.0,0.043653
195,서울특별시 송파구,철도,3687.0,139211.0,0.026485
183,서울특별시 송파구,업무시설,3375.0,139211.0,0.024244


In [15]:
## blockType별 평균 인구량 계산
# 거주인구 분배를 위해 단순 총합뿐 아니라, 해당 blockType을 가진
# 격자 하나당 평균적의 인구 배분을 파악한다.
block_pop_mean_city = (
    grid_pop_lu
    .groupby(["gbn", "blockType"], as_index=False)["total_pop"]
    .mean()
    .rename(columns={"total_pop": "block_pop_mean"})
)

display(block_pop_mean_city.head(20))

,gbn,blockType,block_pop_mean
0,경기도 성남시,가스공급설비,0.000000
1,경기도 성남시,가압장,1.529412
2,경기도 성남시,경관녹지,19.037500
3,경기도 성남시,고등학교,86.785714
4,경기도 성남시,공공공지,71.321543
5,경기도 성남시,공공청사,0.000000
6,경기도 성남시,공원,43.658031
7,경기도 성남시,광장,80.130435
8,경기도 성남시,교통광장,14.459016
9,경기도 성남시,근린공원,39.564706


In [16]:
## 저장용 학습 데이터셋 생성
# 최종적으로 도시별 'blockType별' 인구 총합, 도시 전체 인구, 비중, 평균 인구량을 정리한 데이터셋 생성
# 격자 단위의 인구를 재구성하는 단계의 기반 자료로 사용
resident_blocktype_profile = block_pop_city.merge(
    block_pop_mean_city,
    on=["gbn", "blockType"],
    how="left"
)

display(resident_blocktype_profile.head(20))

,gbn,blockType,block_pop_sum,city_pop_sum,block_pop_ratio,block_pop_mean
0,경기도 성남시,가스공급설비,0.0,410595.0,0.000000,0.000000
1,경기도 성남시,가압장,26.0,410595.0,0.000063,1.529412
2,경기도 성남시,경관녹지,1523.0,410595.0,0.003709,19.037500
3,경기도 성남시,고등학교,2430.0,410595.0,0.005918,86.785714
4,경기도 성남시,공공공지,44362.0,410595.0,0.108043,71.321543
5,경기도 성남시,공공청사,0.0,410595.0,0.000000,0.000000
6,경기도 성남시,공원,8426.0,410595.0,0.020521,43.658031
7,경기도 성남시,광장,1843.0,410595.0,0.004489,80.130435
8,경기도 성남시,교통광장,882.0,410595.0,0.002148,14.459016
9,경기도 성남시,근린공원,16815.0,410595.0,0.040953,39.564706


In [17]:
## 결과 저장
save_fp = BASE_DIR / "01-01.거주인구_관련_부분.csv"

resident_blocktype_profile.to_csv(
    save_fp,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", save_fp)
print("shape:", resident_blocktype_profile.shape)

저장 완료: /content/drive/MyDrive/신도시공모전/Population Dataset/01-01.거주인구_관련_부분.csv
shape: (200, 6)


## 실제 신도시 인구를 기준으로 격자별 인구 재분배

다음은 위에서 추출한 데이터셋을 통해 **가중치**를 명확히 보고 아래의 3가지의 방법을 수행한다.

1. 신도시 내 blockType 비중
2. 같은 blockType 안에서 각 격자의 상대 비중
3. 최종 격자 배분 인구

In [19]:
## 실제 신도시 인구를 반영한 격자 단위 재분배
# 1. 신도시 내 blockType 비중
# 2. 같은 blockType 내부의 격자 비중
# 단순 균등 분배가 아닌 실제 관측 토지이용-인구 관계를 반영한 격자별 인구 재구성

# grid_pop_lu에 신도시 이름 부여
def classify_newtown(zone_name):
    z = str(zone_name)

    if "동탄" in z:
        return "동탄신도시"
    elif "위례" in z:
        return "위례신도시"
    elif "미사" in z:
        return "하남미사신도시"
    elif "판교" in z:
        return "판교신도시"
    else:
        return np.nan

grid_pop_lu["newtown"] = grid_pop_lu["zoneName"].apply(classify_newtown)

print("신도시별 행 수")
display(grid_pop_lu["newtown"].value_counts(dropna=False))

신도시별 행 수


,count
newtown,
동탄신도시,14993
판교신도시,4536
위례신도시,4120
하남미사신도시,2954


*실제 통계자료 기준 신도시 배분*

- 동탄 신도시 : 40,6769명
- 위례 신도시 : 12,6186명
- 하남미사 신도시 : 130615명,
- 판교 : 87000명

참고한 자료는 **PPT 자료**에 첨부한다.

In [20]:
target_pop = {
    "동탄신도시": 406769,
    "위례신도시": 126186,
    "하남미사신도시": 130615,
    "판교신도시": 87000
}

target_df = pd.DataFrame(
    [(k, v) for k, v in target_pop.items()],
    columns=["newtown", "target_population"]
)

display(target_df)

,newtown,target_population
0,동탄신도시,406769
1,위례신도시,126186
2,하남미사신도시,130615
3,판교신도시,87000


In [21]:
## 신도시별 현재 총합 확인
current_newtown_pop = (
    grid_pop_lu
    .groupby("newtown", as_index=False)["total_pop"]
    .sum()
    .rename(columns={"total_pop": "current_population"})
)

compare_newtown = target_df.merge(current_newtown_pop, on="newtown", how="left")
compare_newtown["scale_factor"] = (
    compare_newtown["target_population"] / compare_newtown["current_population"]
)

print("신도시별 현재 총합 vs 목표 인구")
display(compare_newtown)

신도시별 현재 총합 vs 목표 인구


,newtown,target_population,current_population,scale_factor
0,동탄신도시,406769,1012887.0,0.401594
1,위례신도시,126186,373765.0,0.337608
2,하남미사신도시,130615,419844.0,0.311104
3,판교신도시,87000,271829.0,0.320054


In [22]:
## 목표인구와 현재 총합 비교
# 기존 거주인구 데이터셋과 격자 총합의 실제 신도시 계획 인구를 비교해 스케일 차이 확인
# blockType 별 인구 총합 및 비중 계산
# 신도시-블록타입별 총 인구
block_sum_nt = (
    grid_pop_lu
    .groupby(["newtown", "blockType"], as_index=False)["total_pop"]
    .sum()
    .rename(columns={"total_pop": "block_pop_sum"})
)

# 신도시 총합
newtown_sum = (
    block_sum_nt
    .groupby("newtown", as_index=False)["block_pop_sum"]
    .sum()
    .rename(columns={"block_pop_sum": "newtown_pop_sum"})
)

# blockType 비중
block_sum_nt = block_sum_nt.merge(newtown_sum, on="newtown", how="left")
block_sum_nt["block_weight"] = block_sum_nt["block_pop_sum"] / block_sum_nt["newtown_pop_sum"]

print("신도시별 blockType 비중")
display(block_sum_nt.head(20))

신도시별 blockType 비중


,newtown,blockType,block_pop_sum,newtown_pop_sum,block_weight
0,동탄신도시,가스공급설비,0.0,1012887.0,0.000000
1,동탄신도시,가압장,0.0,1012887.0,0.000000
2,동탄신도시,공공공지,72061.0,1012887.0,0.071144
3,동탄신도시,공공청사,1212.0,1012887.0,0.001197
4,동탄신도시,공동주택,200013.0,1012887.0,0.197468
5,동탄신도시,공영차고지,0.0,1012887.0,0.000000
6,동탄신도시,공원,86509.0,1012887.0,0.085408
7,동탄신도시,광장,8041.0,1012887.0,0.007939
8,동탄신도시,교육시설,664.0,1012887.0,0.000656
9,동탄신도시,교통광장,961.0,1012887.0,0.000949


### 1차 가중치 : blockType 비중

각 신도시에서 'blockType'별 총 인구를 계산하여 신도시 전체 인구 대비 비중을 구하였다.

In [23]:
## 같은 blockType 내부의 격자 비중 계산
# 격자 중복 방지용: gid 단위 정리
grid_base = (
    grid_pop_lu[["gid", "gbn", "newtown", "blockType", "total_pop"]]
    .drop_duplicates()
    .copy()
)

# 같은 신도시-같은 blockType 내부 총합
grid_base["block_group_sum"] = (
    grid_base
    .groupby(["newtown", "blockType"])["total_pop"]
    .transform("sum")
)

# 같은 blockType 내부 격자 비중
# total_pop이 모두 0인 경우 균등 분배를 위해 별도 처리
grid_base["grid_weight_in_block"] = np.where(
    grid_base["block_group_sum"] > 0,
    grid_base["total_pop"] / grid_base["block_group_sum"],
    np.nan
)

# block_group_sum == 0 인 경우 균등 분배
group_size = (
    grid_base
    .groupby(["newtown", "blockType"])["gid"]
    .transform("count")
)

grid_base["grid_weight_in_block"] = np.where(
    grid_base["grid_weight_in_block"].isna(),
    1 / group_size,
    grid_base["grid_weight_in_block"]
)

print("격자 내부 비중 확인")
display(grid_base.head(20))

격자 내부 비중 확인


,gid,gbn,newtown,blockType,total_pop,block_group_sum,grid_weight_in_block
1524,다사611327,경기도 성남시,판교신도시,연립,151.0,2598.0,0.058122
1524,다사611327,경기도 성남시,판교신도시,도로,151.0,40972.0,0.003685
1524,다사611327,경기도 성남시,판교신도시,근린공원,151.0,15550.0,0.009711
1525,다사611328,경기도 성남시,판교신도시,연립,7.0,2598.0,0.002694
1525,다사611328,경기도 성남시,판교신도시,근린공원,7.0,15550.0,0.000450
1526,다사611329,경기도 성남시,판교신도시,연립,0.0,2598.0,0.000000
1526,다사611329,경기도 성남시,판교신도시,근린공원,0.0,15550.0,0.000000
1598,다사612323,경기도 성남시,판교신도시,경관녹지,0.0,1523.0,0.000000
1599,다사612324,경기도 성남시,판교신도시,경관녹지,0.0,1523.0,0.000000
1599,다사612324,경기도 성남시,판교신도시,도로,0.0,40972.0,0.000000


#### 2차 가중치 : blockType 내부 격자 비중

같은 'blockType' 이라 하더라고 기존 인구 규모가 다를 수 있기에 동일한 'blockType' 내부에서 개발 격자가 차지하는 상대 비중을 계산하였다.

- 'grid_weight_in_block' 은 1차적으로 배분된 인구를 개별 격자에 다시 분배하는데 사용한다.

In [24]:
## 목표 인구를 격자에 최종 분배
# 목표 인구 결합
alloc_df = grid_base.merge(
    block_sum_nt[["newtown", "blockType", "block_weight"]],
    on=["newtown", "blockType"],
    how="left"
).merge(
    target_df,
    on="newtown",
    how="left"
)

# 최종 배분 인구
alloc_df["allocated_pop"] = (
    alloc_df["target_population"] *
    alloc_df["block_weight"] *
    alloc_df["grid_weight_in_block"]
)

print("최종 배분 결과 샘플")
display(alloc_df.head(20))

최종 배분 결과 샘플


,gid,gbn,newtown,blockType,total_pop,block_group_sum,grid_weight_in_block,block_weight,target_population,allocated_pop
0,다사611327,경기도 성남시,판교신도시,연립,151.0,2598.0,0.058122,0.010234,87000,51.750958
1,다사611327,경기도 성남시,판교신도시,도로,151.0,40972.0,0.003685,0.152379,87000,48.857791
2,다사611327,경기도 성남시,판교신도시,근린공원,151.0,15550.0,0.009711,0.061859,87000,52.259697
3,다사611328,경기도 성남시,판교신도시,연립,7.0,2598.0,0.002694,0.010234,87000,2.399051
4,다사611328,경기도 성남시,판교신도시,근린공원,7.0,15550.0,0.000450,0.061859,87000,2.422635
5,다사611329,경기도 성남시,판교신도시,연립,0.0,2598.0,0.000000,0.010234,87000,0.000000
6,다사611329,경기도 성남시,판교신도시,근린공원,0.0,15550.0,0.000000,0.061859,87000,0.000000
7,다사612323,경기도 성남시,판교신도시,경관녹지,0.0,1523.0,0.000000,0.005603,87000,0.000000
8,다사612324,경기도 성남시,판교신도시,경관녹지,0.0,1523.0,0.000000,0.005603,87000,0.000000
9,다사612324,경기도 성남시,판교신도시,도로,0.0,40972.0,0.000000,0.152379,87000,0.000000


In [25]:
## 최종 합계 검증 단게
final_check = (
    alloc_df
    .groupby("newtown", as_index=False)["allocated_pop"]
    .sum()
    .merge(target_df, on="newtown", how="left")
)

final_check["diff"] = final_check["allocated_pop"] - final_check["target_population"]

print("신도시별 최종 합계 검증")
display(final_check)

신도시별 최종 합계 검증


,newtown,allocated_pop,target_population,diff
0,동탄신도시,406769.0,406769,0.0
1,위례신도시,126186.0,126186,0.0
2,판교신도시,87000.0,87000,0.0
3,하남미사신도시,130615.0,130615,0.0


#### 최종 분배
이로써 1차 가중치 + 2차 가중치를 통해 각 신도시별 목표인구를 겨자 단위로 최종 배분하였다.

이후 신도시별로 배분 인구 총합이 일치하는지 확인하여 가중치 기반 인구 분배 수행 여부를 확인했다.

In [26]:
## 가중치 확인용 테이블 저장 코드
weight_check_fp = BASE_DIR / "01-02.거주인구_가중치확인.csv"

alloc_df[[
    "gid", "gbn", "newtown", "blockType",
    "total_pop", "block_weight", "grid_weight_in_block",
    "target_population", "allocated_pop"
]].to_csv(
    weight_check_fp,
    index=False,
    encoding="utf-8-sig"
)

print("가중치 확인 파일 저장 완료:", weight_check_fp)

가중치 확인 파일 저장 완료: /content/drive/MyDrive/신도시공모전/Population Dataset/01-02.거주인구_가중치확인.csv


In [27]:
## 최종 신도시 격자 인구 저장
final_city_grid_fp = BASE_DIR / "01-02.신도시별_격자_거주인구.csv"

alloc_df[[
    "gid", "gbn", "newtown", "blockType", "allocated_pop"
]].rename(columns={
    "allocated_pop": "resident_pop_reallocated"
}).to_csv(
    final_city_grid_fp,
    index=False,
    encoding="utf-8-sig"
)

print("최종 신도시 격자 인구 저장 완료:", final_city_grid_fp)

최종 신도시 격자 인구 저장 완료: /content/drive/MyDrive/신도시공모전/Population Dataset/01-02.신도시별_격자_거주인구.csv


## 하남교산에 거주인구 데이터셋 적용해보기

하남교산은 이론적으로 **87258**의 인구가 수용 가능하다고 한다.

기존의 4개의 신도시는 거주인구 데이터셋이 존재했기에 토지이용별로 거주인구의 분포도를 확인할 수 있었다.
하지만 하남교산은 아직 신도시 거주 이전이기에 거주인구의 데이터셋을 확보할 수가 없다.

따라서 아래와 같은 분석 목적을 취했다.

1. 격자별 총 거주인구 계산
2. 4개 신도시 토지이용계획도와 join한 'blockType'별 **거주인구 총합, 평균, 전체 대비 가중**을 계산한다.
3. 이를 기반으로 향후 하남교산 신도시에 적용할 거주인구 가중치를 근거로 확인한다.

In [28]:
## 토지이용 유형별 거주인구 분포 특성 학습하기
BASE_DIR = Path("/content/drive/MyDrive/신도시공모전/Population Dataset")

grid_fp = BASE_DIR / "01._격자_(4개_시·구).geojson"
pop_fp = BASE_DIR / "03._성연령별_거주인구(격자).csv"
landuse_fp = BASE_DIR / "22._토지이용계획도_(4개_신도시).geojson"

print("파일 존재 여부")
print("grid   :", grid_fp.exists())
print("pop    :", pop_fp.exists())
print("landuse:", landuse_fp.exists())

파일 존재 여부
grid   : True
pop    : True
landuse: True


In [29]:
## 격자 단위 총 거주인구 계산
grid = gpd.read_file(grid_fp)
pop = pd.read_csv(pop_fp)
landuse = gpd.read_file(landuse_fp)

print("grid shape   :", grid.shape)
print("pop shape    :", pop.shape)
print("landuse shape:", landuse.shape)

print("\ngrid columns")
print(grid.columns.tolist())

print("\npop columns")
print(pop.columns.tolist())

print("\nlanduse columns")
print(landuse.columns.tolist())

grid shape   : (99323, 4)
pop shape    : (99323, 21)
landuse shape: (5337, 5)

grid columns
['std_yr', 'gbn', 'gid', 'geometry']

pop columns
['gbn', 'gid', 'year', 'm_20g_pop', 'w_20g_pop', 'm_30g_pop', 'w_30g_pop', 'm_40g_pop', 'w_40g_pop', 'm_50g_pop', 'w_50g_pop', 'm_60g_pop', 'w_60g_pop', 'm_70g_pop', 'w_70g_pop', 'm_80g_pop', 'w_80g_pop', 'm_90g_pop', 'w_90g_pop', 'm_100g_pop', 'w_100g_pop']

landuse columns
['zoneCode', 'zoneName', 'blockName', 'blockType', 'geometry']


In [30]:
## 격자와 거주인구 결합
pop_cols = [c for c in pop.columns if c.endswith("_pop")]

pop[pop_cols] = pop[pop_cols].fillna(0)
pop["total_pop"] = pop[pop_cols].sum(axis=1)

print("총 거주인구 합계:", pop["total_pop"].sum())
display(pop[["gid", "total_pop"]].head())

총 거주인구 합계: 2183388.0


,gid,total_pop
0,다사581304,0.0
1,다사581305,0.0
2,다사581306,0.0
3,다사582304,0.0
4,다사582305,0.0


In [31]:
grid_pop = grid.merge(
    pop[["gid", "total_pop"]],
    on="gid",
    how="left"
)

grid_pop["total_pop"] = grid_pop["total_pop"].fillna(0)

print("결합 후 shape:", grid_pop.shape)
print("total_pop 결측 개수:", grid_pop["total_pop"].isna().sum())
display(grid_pop[["gid", "gbn", "total_pop"]].head())

결합 후 shape: (99679, 5)
total_pop 결측 개수: 0


,gid,gbn,total_pop
0,다사581304,경기도 성남시,0.0
1,다사581305,경기도 성남시,0.0
2,다사581306,경기도 성남시,0.0
3,다사582304,경기도 성남시,0.0
4,다사582305,경기도 성남시,0.0


In [32]:
## 좌표계 통일 및 토지이용계획도 매핑
if grid_pop.crs is None:
    grid_pop = grid_pop.set_crs("EPSG:4326")
else:
    grid_pop = grid_pop.to_crs("EPSG:4326")

if landuse.crs is None:
    landuse = landuse.set_crs("EPSG:4326")
else:
    landuse = landuse.to_crs("EPSG:4326")

use_cols = [c for c in ["blockType", "zoneName", "geometry"] if c in landuse.columns]

grid_pop_lu = gpd.sjoin(
    grid_pop,
    landuse[use_cols],
    how="inner",
    predicate="intersects"
).drop(columns=["index_right"], errors="ignore")

print("매핑 후 shape:", grid_pop_lu.shape)
print("blockType 결측 개수:", grid_pop_lu["blockType"].isna().sum())
print("blockType 종류 수:", grid_pop_lu["blockType"].nunique())

display(grid_pop_lu[["gid", "gbn", "blockType", "total_pop"]].head())

매핑 후 shape: (26603, 7)
blockType 결측 개수: 0
blockType 종류 수: 88


,gid,gbn,blockType,total_pop
1524,다사611327,경기도 성남시,연립,151.0
1524,다사611327,경기도 성남시,도로,151.0
1524,다사611327,경기도 성남시,근린공원,151.0
1525,다사611328,경기도 성남시,연립,7.0
1525,다사611328,경기도 성남시,근린공원,7.0


In [33]:
## blockType별 거주인구 총합, 평균, 비중 계싼

# blockType별 총합
block_sum = (
    grid_pop_lu
    .groupby("blockType", as_index=False)["total_pop"]
    .sum()
    .rename(columns={"total_pop": "block_pop_sum"})
)

# blockType별 평균
block_mean = (
    grid_pop_lu
    .groupby("blockType", as_index=False)["total_pop"]
    .mean()
    .rename(columns={"total_pop": "block_pop_mean"})
)

# 전체 대비 비중
total_sum = block_sum["block_pop_sum"].sum()
block_sum["block_pop_ratio"] = block_sum["block_pop_sum"] / total_sum

# 통합
block_profile = block_sum.merge(block_mean, on="blockType", how="left")

# 정렬
block_profile = block_profile.sort_values("block_pop_sum", ascending=False).reset_index(drop=True)

display(block_profile.head(30))

,blockType,block_pop_sum,block_pop_ratio,block_pop_mean
0,아파트,368423.0,0.177269,191.587624
1,도로,347805.0,0.167349,84.768462
2,공동주택,200013.0,0.096238,143.790798
3,단독주택,142296.0,0.068467,49.928421
4,공공공지,129460.0,0.062291,65.782520
5,보행자전용도로,120075.0,0.057775,97.226721
6,공원,112729.0,0.054240,47.605152
7,녹지,93057.0,0.044775,59.121347
8,완충녹지,82278.0,0.039589,116.872159
9,학교,39285.0,0.018902,73.843985


세 가지 지표 설명

- block_pop_sum : 해당 blockType의 총 거주인구
- block_pop_mean : 해당 blockType 격자들의 평균 거주인구
- block_pop_ratio : 전체 거주인구 대비 해당 blockType의 비중

In [35]:
## 도시별 차이 확인하기
# 거주인구의 규모가 다르다는 것을 반영하여 도시별 평균 거주인구도 함께 확인한다.
block_city_mean = (
    grid_pop_lu
    .groupby(["gbn", "blockType"], as_index=False)["total_pop"]
    .mean()
    .rename(columns={"total_pop": "block_pop_mean"})
)

display(block_city_mean.head(30))

,gbn,blockType,block_pop_mean
0,경기도 성남시,가스공급설비,0.000000
1,경기도 성남시,가압장,1.529412
2,경기도 성남시,경관녹지,19.037500
3,경기도 성남시,고등학교,86.785714
4,경기도 성남시,공공공지,71.321543
5,경기도 성남시,공공청사,0.000000
6,경기도 성남시,공원,43.658031
7,경기도 성남시,광장,80.130435
8,경기도 성남시,교통광장,14.459016
9,경기도 성남시,근린공원,39.564706


In [36]:
## 가중치 확인용 결과 저장
save_fp_1 = BASE_DIR / "01-02.blockType별_거주인구_가중치.csv"
save_fp_2 = BASE_DIR / "01-02.blockType별_도시별_평균거주인구.csv"

block_profile.to_csv(save_fp_1, index=False, encoding="utf-8-sig")
block_city_mean.to_csv(save_fp_2, index=False, encoding="utf-8-sig")

print("저장 완료:")
print(save_fp_1)
print(save_fp_2)

저장 완료:
/content/drive/MyDrive/신도시공모전/Population Dataset/01-02.blockType별_거주인구_가중치.csv
/content/drive/MyDrive/신도시공모전/Population Dataset/01-02.blockType별_도시별_평균거주인구.csv


### 하남교산 적용 전 blockType 가중치 보정

4개 신도시 기존 거주인구데이터와 토지이용계획도를 결합한 결과, 이미 거주가 형성된 실제 도시의 격자 기반 결과이기에, 도로, 공원, 공공공지와 같은 비주거 용도 격자에도 생활권 효과로 인해 일정 수준의 인구가 포함되어 있었다.

따라서 하남교산에는 'block_pop_mean'을 그대로 사용하지 않고,
토지이용의 실제 거주 가능성을 반영한 **보정 계수(residential suitability factor)**를 추가 적용하였다.

- 아파트, 공동주택, 단독주택, 주상복합 등 **직접 주거 가능**이 강한 유형은 높은 가중치
- 근린생활시설, 준주거, 복합용지 등 **혼합 기능**은 유형은 중간 가중치
- 도로, 공원, 녹지, 하천, 학교, 공공시설 등 **비주거 기능** 유형은 0 또는 매우 낮은 가중치

하남교산 목표인구 **87,258명**을 기준으로 전체 격자 인구기준 총합이 일치하도록 스케일링 진행

In [37]:
BASE_DIR = Path("/content/drive/MyDrive/신도시공모전/Population Dataset")

grid_gyosan_fp = BASE_DIR / "02._격자_(하남교산).geojson"
landuse_gyosan_fp = BASE_DIR / "23._토지이용계획도_(하남교산).geojson"

grid_gyosan = gpd.read_file(grid_gyosan_fp)
landuse_gyosan = gpd.read_file(landuse_gyosan_fp)

print("grid_gyosan shape:", grid_gyosan.shape)
print("landuse_gyosan shape:", landuse_gyosan.shape)

print("\ngrid_gyosan columns")
print(grid_gyosan.columns.tolist())

print("\nlanduse_gyosan columns")
print(landuse_gyosan.columns.tolist())

grid_gyosan shape: (770, 3)
landuse_gyosan shape: (761, 5)

grid_gyosan columns
['gbn', 'gid', 'geometry']

landuse_gyosan columns
['zoneCode', 'zoneName', 'blockName', 'blockType', 'geometry']


In [38]:
# 좌표계 통일
if grid_gyosan.crs is None:
    grid_gyosan = grid_gyosan.set_crs("EPSG:4326")
else:
    grid_gyosan = grid_gyosan.to_crs("EPSG:4326")

if landuse_gyosan.crs is None:
    landuse_gyosan = landuse_gyosan.set_crs("EPSG:4326")
else:
    landuse_gyosan = landuse_gyosan.to_crs("EPSG:4326")

print("grid_gyosan CRS:", grid_gyosan.crs)
print("landuse_gyosan CRS:", landuse_gyosan.crs)

grid_gyosan CRS: EPSG:4326
landuse_gyosan CRS: EPSG:4326


In [39]:
# 하남교산 격자에 blockType 매핑
use_cols = [c for c in ["blockType", "zoneName", "geometry"] if c in landuse_gyosan.columns]

gyosan_lu = gpd.sjoin(
    grid_gyosan,
    landuse_gyosan[use_cols],
    how="left",
    predicate="intersects"
).drop(columns=["index_right"], errors="ignore")

print("매핑 후 shape:", gyosan_lu.shape)
print("blockType 결측 개수:", gyosan_lu["blockType"].isna().sum())
print("blockType 종류 수:", gyosan_lu["blockType"].nunique())

display(gyosan_lu[["gid", "gbn", "blockType"]].head())

매핑 후 shape: (3557, 5)
blockType 결측 개수: 0
blockType 종류 수: 37


,gid,gbn,blockType
0,다사720443,경기도 하남시,단독주택
0,다사720443,경기도 하남시,경관녹지
0,다사720443,경기도 하남시,도로
1,다사720444,경기도 하남시,단독주택
1,다사720444,경기도 하남시,단독주택


In [40]:
## blockType 보정 계수 정의
# block_profile은 이전 단계에서 이미 생성된 결과 사용
# 컬럼: blockType, block_pop_sum, block_pop_ratio, block_pop_mean

# 보정 계수 정의
res_factor_map = {
    # 주거 핵심
    "아파트": 1.0,
    "공동주택": 1.0,
    "단독주택": 0.8,
    "연립": 0.8,
    "주상복합": 0.75,
    "준주거용지": 0.6,

    # 일부 거주 가능
    "근린생활시설용지": 0.35,
    "근린상업": 0.20,
    "중심상업": 0.15,
    "일반상업": 0.15,
    "상업시설": 0.15,
    "복합용지": 0.30,
    "업무시설": 0.10,

    # 사실상 비주거
    "도로": 0.0,
    "보행자전용도로": 0.0,
    "공공공지": 0.0,
    "공원": 0.0,
    "녹지": 0.0,
    "완충녹지": 0.0,
    "연결녹지": 0.0,
    "근린공원": 0.0,
    "소공원": 0.0,
    "경관녹지": 0.0,
    "하천": 0.0,
    "광장": 0.0,
    "주차장": 0.0,
    "학교": 0.0,
    "초등학교": 0.0,
    "중학교": 0.0,
    "고등학교": 0.0,
    "유치원": 0.0,
    "공공청사": 0.0,
    "도시지원시설용지": 0.0,
    "수도공급시설": 0.0,
    "가스공급설비": 0.0,
    "가압장": 0.0,
    "변전시설": 0.0,
    "배수시설": 0.0,
    "교통광장": 0.0,
}

# block_profile에 보정 계수 추가
block_weight_table = block_profile.copy()
block_weight_table["res_factor"] = block_weight_table["blockType"].map(res_factor_map).fillna(0.0)

# 최종 초기 가중치
block_weight_table["adjusted_weight"] = (
    block_weight_table["block_pop_mean"] * block_weight_table["res_factor"]
)

display(
    block_weight_table[["blockType", "block_pop_mean", "res_factor", "adjusted_weight"]]
    .sort_values("adjusted_weight", ascending=False)
    .head(30)
)

,blockType,block_pop_mean,res_factor,adjusted_weight
0,아파트,191.587624,1.00,191.587624
2,공동주택,143.790798,1.00,143.790798
12,주상복합,154.473029,0.75,115.854772
20,준주거용지,128.934959,0.60,77.360976
10,중심상업,322.801653,0.15,48.420248
24,연립,50.699552,0.80,40.559641
3,단독주택,49.928421,0.80,39.942737
11,일반상업,173.206422,0.15,25.980963
31,복합용지,80.323529,0.30,24.097059
18,근린생활시설용지,42.941176,0.35,15.029412


#### 초기 가중치 계산

`adjusted_weight = block_pop_mean X res_factor`

- 실제 도시에서 관측된 blockType별 평균 거주인구에 계획 도시 적용을 위한 보정계수를 곱하여 하남교산 적용용 blockType 가중치 생성

In [41]:
# 하남교산 격자에 가중치 결합
gyosan_alloc = gyosan_lu.merge(
    block_weight_table[["blockType", "block_pop_mean", "res_factor", "adjusted_weight"]],
    on="blockType",
    how="left"
)

gyosan_alloc["adjusted_weight"] = gyosan_alloc["adjusted_weight"].fillna(0)

print("가중치 결합 후")
display(gyosan_alloc[["gid", "blockType", "block_pop_mean", "res_factor", "adjusted_weight"]].head(20))

가중치 결합 후


,gid,blockType,block_pop_mean,res_factor,adjusted_weight
0,다사720443,단독주택,49.928421,0.8,39.942737
1,다사720443,경관녹지,43.807107,0.0,0.000000
2,다사720443,도로,84.768462,0.0,0.000000
3,다사720444,단독주택,49.928421,0.8,39.942737
4,다사720444,단독주택,49.928421,0.8,39.942737
5,다사720444,경관녹지,43.807107,0.0,0.000000
6,다사720444,도로,84.768462,0.0,0.000000
7,다사721443,하천,22.429299,0.0,0.000000
8,다사721443,경관녹지,43.807107,0.0,0.000000
9,다사721443,단독주택,49.928421,0.8,39.942737


In [42]:
# 전체 목표 인구
TARGET_POP_GYOSAN = 87258

# 가중치 총합
weight_sum = gyosan_alloc["adjusted_weight"].sum()

print("가중치 총합:", weight_sum)

if weight_sum == 0:
    raise ValueError("adjusted_weight 총합이 0입니다. blockType 명칭 또는 보정 계수 확인 필요")

# 가중치 비율
gyosan_alloc["weight_ratio"] = gyosan_alloc["adjusted_weight"] / weight_sum

# 최종 인구 배분
gyosan_alloc["resident_pop"] = TARGET_POP_GYOSAN * gyosan_alloc["weight_ratio"]

print("최종 배분 합계:", gyosan_alloc["resident_pop"].sum())
display(gyosan_alloc[["gid", "blockType", "weight_ratio", "resident_pop"]].head(20))

가중치 총합: 73209.90314130284
최종 배분 합계: 87258.0


,gid,blockType,weight_ratio,resident_pop
0,다사720443,단독주택,0.000546,47.607266
1,다사720443,경관녹지,0.000000,0.000000
2,다사720443,도로,0.000000,0.000000
3,다사720444,단독주택,0.000546,47.607266
4,다사720444,단독주택,0.000546,47.607266
5,다사720444,경관녹지,0.000000,0.000000
6,다사720444,도로,0.000000,0.000000
7,다사721443,하천,0.000000,0.000000
8,다사721443,경관녹지,0.000000,0.000000
9,다사721443,단독주택,0.000546,47.607266


In [43]:
# blockType별 최종 배분 인구 확인
gyosan_block_summary = (
    gyosan_alloc
    .groupby("blockType", as_index=False)["resident_pop"]
    .sum()
    .sort_values("resident_pop", ascending=False)
)

display(gyosan_block_summary.head(30))

,blockType,resident_pop
4,공동주택,63411.531194
10,단독주택,11663.780166
32,주상복합,9942.185107
8,근린생활시설용지,1379.329836
17,복합용지,430.814904
23,연립,338.397828
20,상업시설,91.960965
1,경관녹지,0.000000
0,가압장,0.000000
3,공공청사,0.000000


In [44]:
## 하남교산에 blockType별 인구 배분 결과 확인
# 최종 저장
save_fp = BASE_DIR / "01-03.하남교산_거주인구_격자배분.csv"

gyosan_alloc[[
    "gid", "gbn", "blockType",
    "block_pop_mean", "res_factor", "adjusted_weight",
    "weight_ratio", "resident_pop"
]].to_csv(
    save_fp,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", save_fp)
print("최종 shape:", gyosan_alloc.shape)

저장 완료: /content/drive/MyDrive/신도시공모전/Population Dataset/01-03.하남교산_거주인구_격자배분.csv
최종 shape: (3557, 10)


In [45]:
# 하남교산 토지이용 면적 계산

lu = landuse_gyosan.copy()

# 면적 계산 (EPSG4326이면 미터 단위 좌표계로 변환)
lu = lu.to_crs(5186)

lu["area_m2"] = lu.geometry.area

block_area = (
    lu.groupby("blockType", as_index=False)["area_m2"]
    .sum()
)

total_area = block_area["area_m2"].sum()

block_area["area_ratio"] = block_area["area_m2"] / total_area

block_area = block_area.sort_values("area_ratio", ascending=False)

display(block_area.head(30))

,blockType,area_m2,area_ratio
4,공동주택,1.325887e+06,0.210073
5,공원,1.262545e+06,0.200037
11,도로,1.008947e+06,0.159857
28,자족기능확보시설,5.677258e+05,0.089950
36,하천,3.096470e+05,0.049060
1,경관녹지,2.940152e+05,0.046584
7,교육시설,2.506837e+05,0.039718
16,복합시설용지기타,2.029988e+05,0.032163
10,단독주택,1.792268e+05,0.028397
32,주상복합,1.729735e+05,0.027406


In [46]:
residential_types = [
    "아파트",
    "공동주택",
    "단독주택",
    "연립",
    "주상복합",
    "준주거용지"
]

block_area[block_area["blockType"].isin(residential_types)]

,blockType,area_m2,area_ratio
4,공동주택,1.325887e+06,0.210073
10,단독주택,1.792268e+05,0.028397
32,주상복합,1.729735e+05,0.027406
23,연립,1.461716e+04,0.002316


In [47]:
# 이전에 만든 인구 배분 결과
pop_result = gyosan_block_summary.copy()

pop_total = pop_result["resident_pop"].sum()

pop_result["pop_ratio"] = pop_result["resident_pop"] / pop_total

compare = block_area.merge(
    pop_result,
    on="blockType",
    how="left"
)

compare = compare[[
    "blockType",
    "area_ratio",
    "resident_pop",
    "pop_ratio"
]].sort_values("area_ratio", ascending=False)

display(compare.head(30))

,blockType,area_ratio,resident_pop,pop_ratio
0,공동주택,0.210073,63411.531194,0.726713
1,공원,0.200037,0.000000,0.000000
2,도로,0.159857,0.000000,0.000000
3,자족기능확보시설,0.089950,0.000000,0.000000
4,하천,0.049060,0.000000,0.000000
5,경관녹지,0.046584,0.000000,0.000000
6,교육시설,0.039718,0.000000,0.000000
7,복합시설용지기타,0.032163,0.000000,0.000000
8,단독주택,0.028397,11663.780166,0.133670
9,주상복합,0.027406,9942.185107,0.113940


In [48]:
compare[
    compare["blockType"].isin(residential_types)
]

,blockType,area_ratio,resident_pop,pop_ratio
0,공동주택,0.210073,63411.531194,0.726713
8,단독주택,0.028397,11663.780166,0.133670
9,주상복합,0.027406,9942.185107,0.113940
24,연립,0.002316,338.397828,0.003878


## 결과 해석

하남교산 격자와 토지이용계획도를 결합한 뒤,  
4개 신도시에서 학습한 `blockType`별 평균 거주인구와 계획도시 적용용 보정 계수를 함께 사용하여  
격자 단위 거주인구를 배분하였다.

이 과정에서 주거 기능이 강한 `blockType`(예: 아파트, 공동주택, 단독주택, 주상복합)은 높은 가중치를 부여받았고,  
비주거 기능이 강한 `blockType`(예: 도로, 공원, 녹지, 학교, 하천 등)은 0 또는 매우 낮은 가중치를 부여받았다.

최종적으로 하남교산의 목표 인구 87,258명을 기준으로 전체 격자 인구 총합이 일치하도록 정규화하였으며,  
이를 통해 토지이용 구조를 반영한 하남교산 격자 단위 거주인구 데이터셋을 구축하였다.